# 금연존 지킴이 — YOLOv8n 담배 감지 모델 학습

**시작 전 필수 설정**: 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 → 저장

셀을 위에서부터 순서대로 실행하세요 (Shift+Enter).

| 단계 | 내용 |
|---|---|
| 1 | GPU 확인·패키지 설치 |
| 2 | Roboflow에서 데이터셋 다운로드 |
| 3 | 학습 전 데이터 검수 |
| 4 | 학습 (약 1~2시간) |
| 5 | 결과 확인 |
| 6 | 테스트 영상으로 정성 확인 (선택) |
| 7 | NCNN 변환 (파이 CPU 플랜 B용) |
| 8 | 결과물 백업 — **세션 끝나기 전에 꼭!** |

## 1. GPU 확인 및 패키지 설치

아래 셀 실행 결과에 `Tesla T4`가 보여야 합니다. 안 보이면 런타임 유형을 다시 확인하세요.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics roboflow

## 2. Roboflow에서 데이터셋 다운로드

아래 값 3개를 채워야 합니다. Roboflow 웹에서 프로젝트 → **Versions** → 학습할 버전 선택 → **Download Dataset** → 형식 `YOLOv8` → **Show download code**를 누르면 API 키·워크스페이스명·프로젝트명이 든 코드가 그대로 나오니, 그 값을 복사해오면 됩니다.

⚠️ API 키가 든 노트북을 저장소에 커밋하지 마세요. 공유할 때는 키를 지우고 공유.

In [ ]:
from roboflow import Roboflow

API_KEY   = "여기에_API_키"          # Roboflow 다운로드 코드에서 복사
WORKSPACE = "여기에_워크스페이스명"
PROJECT   = "여기에_프로젝트명"
VERSION   = 1                        # 팀원에게 확인한 버전 번호

rf = Roboflow(api_key=API_KEY)
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")
print("다운로드 위치:", dataset.location)

## 3. 학습 전 데이터 검수

`nc: 2`, `names: ['cigarette', 'smoke']` 인지, 이미지·라벨 수가 맞는지, 샘플 박스가 제대로 그려지는지 확인합니다. 여기서 이상하면 학습하지 말고 라벨링 담당 팀원과 먼저 확인하세요.

In [ ]:
import glob, os, yaml

yaml_path = os.path.join(dataset.location, "data.yaml")
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)
print("클래스 수:", cfg["nc"])
print("클래스 이름:", cfg["names"])   # ['cigarette', 'smoke'] 이어야 함

for split in ["train", "valid", "test"]:
    imgs = glob.glob(f"{dataset.location}/{split}/images/*")
    lbls = glob.glob(f"{dataset.location}/{split}/labels/*.txt")
    print(f"{split}: 이미지 {len(imgs)}장 / 라벨 {len(lbls)}개")

In [ ]:
# 샘플 6장에 라벨 박스를 그려서 눈으로 검수
import cv2, random
import matplotlib.pyplot as plt

COLORS = {0: (255, 80, 80), 1: (80, 160, 255)}  # 0 cigarette=빨강, 1 smoke=파랑
NAMES  = cfg["names"]

samples = random.sample(glob.glob(f"{dataset.location}/train/images/*"), 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, img_path in zip(axes.flat, samples):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl_path = img_path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    if os.path.exists(lbl_path):
        for line in open(lbl_path):
            c, x, y, bw, bh = map(float, line.split())
            x1, y1 = int((x - bw/2) * w), int((y - bh/2) * h)
            x2, y2 = int((x + bw/2) * w), int((y + bh/2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), COLORS[int(c)], 2)
            cv2.putText(img, NAMES[int(c)], (x1, max(y1-5, 12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLORS[int(c)], 2)
    ax.imshow(img); ax.axis("off")
plt.tight_layout(); plt.show()
# 여러 번 실행해서 다른 샘플도 확인해보세요

## 4. 학습

T4 기준 약 1~2시간. 브라우저 탭을 닫지 마세요 (무료 Colab은 탭을 닫으면 세션이 끊길 수 있습니다).

구조 변경 금지 — 표준 YOLOv8n 그대로 학습합니다 (Hailo Model Zoo 호환).

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")   # COCO 사전학습 가중치에서 시작
results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,      # 20 epoch 개선 없으면 조기 종료
    device=0,
    name="cig_v1",
)

## 5. 결과 확인

보는 법:
- **results.png** — loss 곡선이 내려가다 평평해지면 정상 수렴
- **confusion_matrix.png** — cigarette/smoke가 background로 새는 정도 확인
- **val_batch0_pred.jpg** — 모델이 실제로 그린 박스 (정성 확인에 제일 유용)
- 클래스별 지표에서 **smoke의 mAP이 낮은 건 정상**입니다. cigarette 위주로 판단하세요 (mAP50 0.6 이상이면 1차로 양호)

In [ ]:
from IPython.display import Image, display
RUN = "runs/detect/cig_v1"
for f in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    print("="*30, f)
    display(Image(filename=f"{RUN}/{f}", width=900))

In [ ]:
# 클래스별 정량 지표
best = YOLO(f"{RUN}/weights/best.pt")
metrics = best.val(data=yaml_path)

## 6. (선택) 테스트 영상으로 정성 확인

폰으로 찍은 흡연 연출 영상을 업로드해서 실제 우리 환경에서 어떻게 반응하는지 봅니다. 공개 데이터셋과 실환경의 도메인 갭을 눈으로 확인하는 단계 — 자체 촬영 계획의 근거가 됩니다.

⚠️ 팀원 얼굴이 나온 영상은 확인 후 Colab에서 삭제하고, 어디에도 업로드하지 마세요.

In [ ]:
from google.colab import files
up = files.upload()  # 영상 파일 선택
video = list(up.keys())[0]
best.predict(source=video, save=True, conf=0.3)
# 결과: runs/detect/predict/ 폴더에 박스가 그려진 영상 저장 → 좌측 파일탐색기에서 다운로드

## 7. NCNN 변환 — 파이 CPU 플랜 B용

변환된 폴더를 라즈베리파이로 가져가면 Hailo 없이 CPU-only 추론 FPS를 잴 수 있습니다 (3주차 벤치마크 항목).

In [ ]:
best.export(format="ncnn")
# 결과: runs/detect/cig_v1/weights/best_ncnn_model/ 폴더

## 8. 결과물 백업 — 세션 끝나기 전에 꼭!

Colab 디스크는 세션이 끝나면 사라집니다. 아래 셀로 결과를 압축해 다운로드하세요.

받은 후 할 일:
1. `best.pt` → GitHub **Release**에 업로드 (저장소에 커밋 금지!)
2. 지표 스크린샷 → 학습 이슈에 코멘트로 공유
3. 데이터셋은 백업 불필요 — 원본은 Roboflow에 있음

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("cig_v1_results", "zip", RUN)
files.download("cig_v1_results.zip")  # 가중치+그래프+예측 이미지 전부 포함